In [1]:
import pdfplumber
import pandas as pd
import re

def extrair_pitstops_regex(caminho_pdf, caminho_csv_saida):
    print("Lendo o PDF como texto")
    dados_totais = []

    padrao = re.compile(r"^(\d+)\s+(.+?)\s+(\d+)\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s*([\d:\.]+)?$")

    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            for pagina in pdf.pages:
                texto = pagina.extract_text()
                if texto:
                    for linha in texto.split('\n'):
                        match = padrao.match(linha.strip())
                        if match and match.group(5):
                            dados_totais.append([match.group(1), match.group(2).strip(),
                                                 match.group(3), match.group(4), match.group(5)])
    except FileNotFoundError:
        print(f"ALERTA: O arquivo {caminho_pdf} não foi encontrado.")
        return

    if not dados_totais:
        print("Nenhum pit stop encontrado! O padrão do texto pode estar diferente.")
        return

    df_pitstops = pd.DataFrame(dados_totais,
                               columns=['Carro', 'Piloto_Equipe', 'Lap', 'Time of Day', 'Pit Time'])

    # --- Validação da extração ---
    paradas_por_carro = df_pitstops.groupby('Carro').size().sort_values(ascending=False)
    print(f"Extração: {len(df_pitstops)} pit stops de {df_pitstops['Carro'].nunique()} carros.")
    print(f"Paradas por carro — mín: {paradas_por_carro.min()} | máx: {paradas_por_carro.max()}")
    print(paradas_por_carro.to_string())

    df_pitstops.to_csv(caminho_csv_saida, index=False, sep=';', encoding='utf-8-sig')
    print(f"Salvo em: {caminho_csv_saida}")
    return df_pitstops

# --- ÁREA DE EXECUÇÃO ---
arquivo_pdf_pit = '../data/01_raw/pit_curvelo_p1.pdf'
arquivo_csv_pit = '../data/02_interim/dados_pitstops_P1.csv'

extrair_pitstops_regex(arquivo_pdf_pit, arquivo_csv_pit)

Lendo o PDF como texto
Extração: 27 pit stops de 26 carros.
Paradas por carro — mín: 1 | máx: 2
Carro
24     2
0      1
1      1
10     1
12     1
111    1
121    1
18     1
19     1
21     1
27     1
29     1
293    1
33     1
38     1
4      1
444    1
51     1
6      1
7      1
73     1
8      1
80     1
81     1
83     1
90     1
Salvo em: ../data/02_interim/dados_pitstops_P1.csv


,Carro,Piloto_Equipe,Lap,Time of Day,Pit Time
0,0,CACA BUENO SCUDERIA CHIARELLI,7,10:39:18.071,38.672
1,1,FELIPE FRAGA EUROFARMA RC,7,10:39:06.803,37.895
2,4,JULIO CAMPOS TMG RACING,10,10:43:18.484,38.817
3,6,HELIO CASTRONEVES MERCADO LIVRE RACING,11,10:45:03.168,43.160
4,7,SERGIO SETTE CAMARA TEAM RC,10,10:43:17.015,39.632
5,8,RAFAEL SUZUKI SCUDERIA BANDEIRAS,10,10:43:13.426,39.916
6,10,RICARDO ZONTA FULL TIME GAZOO RACING,7,10:39:30.873,39.869
7,12,LUCAS FORESTI VOGEL MOTORSPORT,13,10:47:29.453,39.844
8,18,ALLAM KHODAIR BLAU MOTORSPORT,10,10:43:21.226,37.783
9,19,FELIPE MASSA TMG RACING,15,10:50:14.032,38.174
